# Compute the frequency spectrum of a timeseries

This notebook demonstrates how to compute the discrete Fourier Transform (FFT) of a timeseries using `temporal.fft`. The time dimension is detected automatically. The frequency coordinate is expressed in Hz here because the time coordinate is a datetime, so its spacing is known in seconds; a numeric time axis would instead give frequencies in the reciprocal of its own units. We use a short hourly 2m temperature dataset for two locations (Reading and Vancouver), each with a strong diurnal (24-hour) cycle, to show that the transform and the downstream analysis work when the data contains more than one time-series.


In [ ]:
import numpy as np

from earthkit import data as ekd
from earthkit import transforms as ekt

# Hourly 2m temperature timeseries for two locations (72 hourly steps, 3 days)
ds = ekd.from_source("sample", "era5-timeseries-multiple.nc").to_xarray()
da = ds["t2m"]

da

## Compute the FFT

`temporal.fft` detects the time dimension (`valid_time`) automatically and returns a complex-valued result indexed by a `frequency` dimension. The frequency coordinate is derived from the hourly sampling and expressed in Hz.

In [ ]:
spectrum = ekt.temporal.fft(da)

spectrum

## Identify the dominant period

We keep the positive frequencies and compute the amplitude of each frequency component. `temporal.fft` attaches a convenience `period` coordinate (`1 / frequency`) alongside the frequency coordinate, so we can read the dominant cycle directly instead of converting from Hz by hand. Because the frequencies are in Hz, the period is a `timedelta64` duration. Since the spectrum retains the `location` dimension, we use `argmax` along the `frequency` dimension to find the dominant cycle independently for each time-series.

In [ ]:
# Keep positive frequencies only (drop the zero-frequency/mean component)
positive = spectrum.where(spectrum["frequency"] > 0, drop=True)
amplitude = np.abs(positive)

# The `period` coordinate is a timedelta; express it in hours for readability
period_hours = positive["period"] / np.timedelta64(1, "h")
amplitude = amplitude.assign_coords(period_hours=("frequency", period_hours.data))

# Dominant period for each timeseries (one per location)
dominant_period = period_hours.isel(frequency=amplitude.argmax("frequency"))
for location in dominant_period["location"].values:
    period = float(dominant_period.sel(location=location))
    print(f"{location}: dominant period {period:.1f} hours")

In [ ]:
amplitude.plot.line(x="period_hours", hue="location", marker="o")